# Step 13c -- DAG-Aware Selected-Set Control

Planner + DDQN control on top of the DAG-aware temporal simulator.

Assumes you already uploaded:
- `caresim_colab.zip`
- `data/processed/icu_readmit/rl_dataset_selected.parquet`
- `models/icu_readmit/dagaware_selected_causal/`

Workflow:
1. mount Drive and verify GPU
2. unzip the source bundle
3. run a smoke control pass
4. run planner, DDQN training, and evaluation when ready

Default reward setup matches the earlier control tracks:
- handcrafted severity reward
- terminal readmission reward enabled


In [ ]:
# Cell 1: Mount Drive + check GPU
from google.colab import drive
drive.mount('/content/drive')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU:   {torch.cuda.get_device_name(0)}')
    print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Control runs will be slower on CPU.')


In [ ]:
# Cell 2: Unzip source files + verify inputs
import os, sys, zipfile

DRIVE_ROOT = '/content/drive/MyDrive/CareAI'
ZIP_CANDIDATES = [
    os.path.join(DRIVE_ROOT, 'caresim_colab.zip'),
    '/content/drive/MyDrive/caresim_colab.zip',
]
UNZIP_DIR = '/content/caresim_src'
SRC_PATH = os.path.join(UNZIP_DIR, 'src')
DATA_PATH = os.path.join(DRIVE_ROOT, 'data', 'rl_dataset_selected.parquet')
MODEL_DIR = os.path.join(DRIVE_ROOT, 'models', 'icu_readmit', 'dagaware_selected_causal')
DAGAWARE_MODEL_DIR = MODEL_DIR
CTRL_DIR = os.path.join(DRIVE_ROOT, 'models', 'icu_readmit', 'dagaware_control_selected_causal')
REPORT_DIR = os.path.join(DRIVE_ROOT, 'reports', 'icu_readmit', 'dagaware_control_selected_causal')
TERMINAL_MODEL_DIR = os.path.join(DRIVE_ROOT, 'models', 'icu_readmit', 'terminal_readmit_selected')
USE_SEVERITY_REWARD = True
SEVERITY_MODE = 'handcrafted'
USE_TERMINAL_READMIT_REWARD = True
TERMINAL_REWARD_SCALE = 15.0
SCRIPT = os.path.join(UNZIP_DIR, 'scripts', 'icu_readmit', 'step_13c_dagaware_control.py')

ZIP_PATH = next((p for p in ZIP_CANDIDATES if os.path.exists(p)), None)
if ZIP_PATH is None:
    raise FileNotFoundError('caresim_colab.zip not found. Upload it to Drive first.')

print(f'Unzipping {ZIP_PATH} ...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(UNZIP_DIR)
print(f'Unzipped to {UNZIP_DIR}')

for pkg_dir in [
    os.path.join(SRC_PATH, 'careai'),
    os.path.join(SRC_PATH, 'careai', 'icu_readmit'),
    os.path.join(SRC_PATH, 'careai', 'icu_readmit', 'dagaware'),
]:
    init = os.path.join(pkg_dir, '__init__.py')
    if not os.path.exists(init):
        open(init, 'w').close()

for p in [DATA_PATH, MODEL_DIR, SCRIPT]:
    if not os.path.exists(p):
        raise FileNotFoundError(p)
os.makedirs(CTRL_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
print('Ready.')


In [ ]:
# Cell 3: Smoke control pass
import subprocess

def run_streaming(cmd, env):
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc.returncode

SMOKE_ARGS = [
    sys.executable, '-u', SCRIPT,
    'smoke',
    '--data', DATA_PATH,
    '--model-dir', DAGAWARE_MODEL_DIR,
    '--control-model-dir', CTRL_DIR,
    '--report-dir', REPORT_DIR,
    '--device', device,
    '--use-severity-reward',
    '--severity-mode', SEVERITY_MODE,
    '--use-terminal-readmit-reward',
    '--terminal-model-dir', TERMINAL_MODEL_DIR,
    '--terminal-reward-scale', str(TERMINAL_REWARD_SCALE),
]
rc = run_streaming(SMOKE_ARGS, env={**os.environ, 'PYTHONPATH': SRC_PATH, 'PYTHONUNBUFFERED': '1'})
if rc != 0:
    raise RuntimeError('DAG-aware control smoke FAILED.')
print('Smoke control PASSED.')


In [ ]:
# Cell 4: Planner evaluation
PLANNER_ARGS = [
    sys.executable, '-u', SCRIPT,
    'planner',
    '--data', DATA_PATH,
    '--model-dir', DAGAWARE_MODEL_DIR,
    '--control-model-dir', CTRL_DIR,
    '--report-dir', REPORT_DIR,
    '--device', device,
    '--use-severity-reward',
    '--severity-mode', SEVERITY_MODE,
    '--use-terminal-readmit-reward',
    '--terminal-model-dir', TERMINAL_MODEL_DIR,
    '--terminal-reward-scale', str(TERMINAL_REWARD_SCALE),
]
rc = run_streaming(PLANNER_ARGS, env={**os.environ, 'PYTHONPATH': SRC_PATH, 'PYTHONUNBUFFERED': '1'})
if rc != 0:
    raise RuntimeError('DAG-aware planner evaluation FAILED.')
print('Planner evaluation complete.')


In [ ]:
# Cell 5: Train DDQN
TRAIN_ARGS = [
    sys.executable, '-u', SCRIPT,
    'train-ddqn',
    '--data', DATA_PATH,
    '--model-dir', DAGAWARE_MODEL_DIR,
    '--control-model-dir', CTRL_DIR,
    '--report-dir', REPORT_DIR,
    '--device', device,
    '--use-severity-reward',
    '--severity-mode', SEVERITY_MODE,
    '--use-terminal-readmit-reward',
    '--terminal-model-dir', TERMINAL_MODEL_DIR,
    '--terminal-reward-scale', str(TERMINAL_REWARD_SCALE),
    '--train-episodes', '256',
    '--train-steps', '4000',
    '--batch-size', '64',
]
rc = run_streaming(TRAIN_ARGS, env={**os.environ, 'PYTHONPATH': SRC_PATH, 'PYTHONUNBUFFERED': '1'})
if rc != 0:
    raise RuntimeError('DAG-aware DDQN training FAILED.')
print('DDQN training complete.')


In [ ]:
# Cell 6: Train simple multi-armed bandit baseline
BANDIT_ARGS = [
    sys.executable, '-u', SCRIPT,
    'train-bandit',
    '--data', DATA_PATH,
    '--model-dir', DAGAWARE_MODEL_DIR,
    '--control-model-dir', CTRL_DIR,
    '--report-dir', REPORT_DIR,
    '--device', device,
    '--use-severity-reward',
    '--severity-mode', SEVERITY_MODE,
    '--use-terminal-readmit-reward',
    '--terminal-model-dir', TERMINAL_MODEL_DIR,
    '--terminal-reward-scale', str(TERMINAL_REWARD_SCALE),
    '--train-episodes', '256',
    '--train-steps', '4000',
]
rc = run_streaming(BANDIT_ARGS, env={**os.environ, 'PYTHONPATH': SRC_PATH, 'PYTHONUNBUFFERED': '1'})
if rc != 0:
    raise RuntimeError('DAG-aware bandit training FAILED.')
print('Bandit training complete.')


In [ ]:
# Cell 7: Full evaluation
EVAL_ARGS = [
    sys.executable, '-u', SCRIPT,
    'eval',
    '--data', DATA_PATH,
    '--model-dir', DAGAWARE_MODEL_DIR,
    '--control-model-dir', CTRL_DIR,
    '--report-dir', REPORT_DIR,
    '--device', device,
    '--use-severity-reward',
    '--severity-mode', SEVERITY_MODE,
    '--use-terminal-readmit-reward',
    '--terminal-model-dir', TERMINAL_MODEL_DIR,
    '--terminal-reward-scale', str(TERMINAL_REWARD_SCALE),
    '--ddqn-path', os.path.join(CTRL_DIR, 'ddqn_model.pt'),
    '--bandit-path', os.path.join(CTRL_DIR, 'bandit_model.json'),
]
rc = run_streaming(EVAL_ARGS, env={**os.environ, 'PYTHONPATH': SRC_PATH, 'PYTHONUNBUFFERED': '1'})
if rc != 0:
    raise RuntimeError('DAG-aware full control evaluation FAILED.')
print('Full evaluation complete.')


In [ ]:
# Cell 8: Inspect outputs
for root, dirs, files in os.walk(REPORT_DIR):
    for f in sorted(files):
        path = os.path.join(root, f)
        print(f'{os.path.relpath(path, REPORT_DIR):55s} {os.path.getsize(path)/1024:8.1f} KB')

print('\nControl artifacts:', CTRL_DIR)
